# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hajergafsi/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub

In [5]:
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("HF_TOKEN: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Mid-panel month for ALL contract work. Never the _sample table for label logic:
# _sample IS the final month (June 2026) -- it's the sealed test/outcome window, not a random sample.
MONTH = "2026-03"
FACT = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"
print("Using mid-panel month:", MONTH)
print("Path (adjust if the actual partition layout differs -- check the dataset file listing first):")
print(FACT)

HF_TOKEN: ··········
Using mid-panel month: 2026-03
Path (adjust if the actual partition layout differs -- check the dataset file listing first):
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1. One row means:**
One page, on one specific day, for one client. The raw data is basically: "On March 3rd, this page (for this client) got X views and Y clicks." I group many of these daily rows together into two chunks per page — a "before" chunk and an "after" chunk — to make my prediction and check it.

**2. Which table(s):**
The daily performance table (`fact_content_daily_performance`), but only the March 2026 slice. I'm using one single month to start, not the whole multi-year dataset — that comes later once this small version works. I'm specifically avoiding the "sample" table, because that one happens to *be* the very last month of data (June 2026), which I need to save untouched for a final, honest test later — using it now would be like peeking at the exam before studying.

**3. Time window:**
I pick a pretend "today" — March 15, 2026. Everything from March 1–14 is the "before" period (what I'm allowed to look at). Everything from March 15–31 is the "after" period (what actually happened next, used only to check the answer).

**4. What I'm trying to predict:**
Whether a page's traffic dropped a lot — specifically, if its average daily views in the "after" period are more than 20% lower than in the "before" period, I call that page "declining." That's the thing I'm trying to guess ahead of time.

**5. What I deliberately leave out:**
I never let my prediction use any information from the "after" period — because that's literally what I'm trying to guess, not something I should already know. If I accidentally included "after" numbers as an input, my model would look amazing but would actually just be cheating by reading the answer key. That's the trap the rest of the notebook deliberately demonstrates and then fixes.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every field I touch this week, sorted into exactly one bucket:

**FEATURE — knowable before T (2026-03-15), safe to use:**
- feat_avg_impressions — average daily impressions, feature half only
- feat_avg_clicks — average daily clicks, feature half only
- feat_avg_position — average GSC position, feature half only
- feat_ctr — clicks/impressions, computed from feature-half sums only
- feat_days_seen — count of distinct days with a row in the feature half

**LABEL — the thing being predicted, computed only from the target half:**
- tgt_avg_impressions — average daily impressions, target half (2026-03-15 to 03-31).
  This is NOT a feature; it exists only to compute the label below.
- label_declined — 1 if feat_avg_impressions > 5 AND tgt_avg_impressions dropped more
  than 20% versus the feature half. This is what I'm predicting.

**CONTEXT — for grouping/joining/filtering, never as model inputs:**
- content_hash_id, client_hash_id — pseudonymous IDs, used only to join feat/tgt and
  identify rows, never fed to the model
- ga4_data_available — a filter flag (only ~4.2% of March rows are True this month);
  not used this week since none of my 5 features touch GA4-sourced columns, but any
  future feature using sessions/engagement/scroll_rate must filter on this flag first

**EXCLUDED — with the why:**
- tgt_avg_impressions as a FEATURE (as opposed to as the label source) — this is the
  exact leakage trap demonstrated below: including it as a model input lets the model
  read the answer key directly, jumping the quick score from 0.6893 to 0.9922. Deleted
  after the demonstration; never used as a feature going forward.
- Any other target-half column (e.g. target-half clicks or position) — same reasoning,
  excluded on principle even though I didn't build them this week.

In [10]:
# Build the feature-half / target-half split and the proxy label, entirely from month=2026-03.
T = "2026-03-15"

features_and_label = con.execute(f"""
    WITH feat AS (
        SELECT content_hash_id, client_hash_id,
               AVG(gsc_impressions)      AS feat_avg_impressions,
               AVG(gsc_clicks)           AS feat_avg_clicks,
               AVG(gsc_avg_position)     AS feat_avg_position,
               AVG(gsc_clicks) / NULLIF(AVG(gsc_impressions), 0) AS feat_ctr,
               COUNT(DISTINCT report_date) AS feat_days_seen
        FROM '{FACT}'
        WHERE report_date < DATE '{T}'
        GROUP BY 1, 2
    ),
    tgt AS (
        SELECT content_hash_id, client_hash_id,
               AVG(gsc_impressions) AS tgt_avg_impressions
        FROM '{FACT}'
        WHERE report_date >= DATE '{T}'
        GROUP BY 1, 2
    )
    SELECT f.*, t.tgt_avg_impressions,
           CASE WHEN f.feat_avg_impressions > 5
                 AND t.tgt_avg_impressions < 0.8 * f.feat_avg_impressions
                THEN 1 ELSE 0 END AS label_declined
    FROM feat f
    JOIN tgt t USING (content_hash_id, client_hash_id)
""").df()

print("Shape:", features_and_label.shape)
print("Label positive rate:", features_and_label["label_declined"].mean().round(4))
features_and_label.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (319584, 9)
Label positive rate: 0.0874


,content_hash_id,client_hash_id,feat_avg_impressions,feat_avg_clicks,feat_avg_position,feat_ctr,feat_days_seen,tgt_avg_impressions,label_declined
0,content_67741cce996cfafa,client_62f4a7e64f5e0096,2.714286,0.071429,4.638889,0.026316,14,0.470588,0
1,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,15.071429,0.071429,3.941856,0.004739,14,40.470588,0
2,content_65c50dfe9d87a585,client_62f4a7e64f5e0096,99.571429,0.000000,6.104078,0.000000,14,100.823529,0
3,content_275b6f7f733016d4,client_62f4a7e64f5e0096,31.642857,0.071429,4.399802,0.002257,14,21.588235,1
4,content_4dc944b7d0b65ecc,client_62f4a7e64f5e0096,4.928571,0.000000,4.484066,0.000000,14,3.823529,0


My five features and why each is knowable at the decision moment (T = 2026-03-15):

1. feat_avg_impressions — average daily impressions in [03-01, 03-14], entirely before T.
2. feat_avg_clicks — average daily clicks in the same feature-half window, entirely before T.
3. feat_avg_position — average GSC position in the feature-half window, entirely before T.
4. feat_ctr — clicks/impressions computed only from feature-half sums, entirely before T.
5. feat_days_seen — how many distinct days in the feature half this page had any row at
   all, entirely before T.

None of these touch report_date >= T, so none of them can see into the target window.

In [11]:
# HONEST quick score: logistic regression on the 5 legitimate features only.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

d = features_and_label.dropna()
honest_features = ["feat_avg_impressions", "feat_avg_clicks", "feat_avg_position",
                    "feat_ctr", "feat_days_seen"]
X = d[honest_features]
y = d["label_declined"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
auc_honest = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])
print("HONEST quick score (5 legitimate features), ROC-AUC:", round(auc_honest, 4))

HONEST quick score (5 legitimate features), ROC-AUC: 0.6893


In [12]:
# THE TRAP -- add ONE label-derived column on purpose.
# tgt_avg_impressions is LITERALLY what the label was computed from -- pure leakage.
d_leak = d.copy()
leak_features = honest_features + ["tgt_avg_impressions"]  # <-- the deliberate leak

X_leak = d_leak[leak_features]
y_leak = d_leak["label_declined"]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak, y_leak, test_size=0.3, random_state=0, stratify=y_leak)
model_leak = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
auc_leak = roc_auc_score(y_test_l, model_leak.predict_proba(X_test_l)[:, 1])

print("LEAKY quick score (adds tgt_avg_impressions), ROC-AUC:", round(auc_leak, 4))
print("Jump vs honest score:", round(auc_leak - auc_honest, 4))
print("\n-> This jump toward ~1.0 is not a better model. It's the model reading the answer")
print("   key, because tgt_avg_impressions is literally the quantity the label thresholds.")

LEAKY quick score (adds tgt_avg_impressions), ROC-AUC: 0.9922
Jump vs honest score: 0.3029

-> This jump toward ~1.0 is not a better model. It's the model reading the answer
   key, because tgt_avg_impressions is literally the quantity the label thresholds.


In [13]:
# DELETE the leak. The honest number is the one that counts, and the one I report.
final_features = honest_features  # tgt_avg_impressions removed
print("Final feature list (leak removed):", final_features)
print("Honest ROC-AUC I am keeping and reporting:", round(auc_honest, 4))

Final feature list (leak removed): ['feat_avg_impressions', 'feat_avg_clicks', 'feat_avg_position', 'feat_ctr', 'feat_days_seen']
Honest ROC-AUC I am keeping and reporting: 0.6893


Record here after running: honest AUC = 0.6893, leaky AUC = 0.9922, jump = 0.3029.
The leaky number is deleted from every downstream notebook — only the honest AUC and
the 5 honest features carry forward.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# QUERY 1 -- GRAIN: one row really is (report_date, client_hash_id, content_hash_id).
# Zero rows back means the grain holds.
grain_check = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM '{FACT}'
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print("Grain violations (expect 0 rows):")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (expect 0 rows):


,report_date,client_hash_id,content_hash_id,c


In [7]:
# QUERY 2 -- ROW COUNT + DATE SPAN for my lane's slice (month = 2026-03).
count_span = con.execute(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM '{FACT}'
""").df()
print("Row count + date span for month=2026-03:")
count_span

Row count + date span for month=2026-03:


,n_rows,n_content,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [9]:
# QUERY 3 -- AVAILABILITY: filter with IS TRUE, show how many rows survive.
# ga4_data_available = FALSE means "not tracked yet for this client", not "zero engagement" --
# so any feature that touches GA4 columns (sessions, engagement_rate, scroll_rate) must be
# computed only on rows where this flag IS TRUE.
availability = con.execute(f"""
    SELECT ga4_data_available, COUNT(*) AS n_rows
    FROM '{FACT}'
    GROUP BY 1
""").df()
print("Split by availability flag:")
print(availability)

survivors = con.execute(f"""
    SELECT COUNT(*) AS n_rows_with_ga4
    FROM '{FACT}'
    WHERE ga4_data_available IS TRUE
""").df()
print("\nRows surviving the IS TRUE filter (GA4-eligible rows):")
survivors

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Split by availability flag:
   ga4_data_available   n_rows
0                <NA>  3018741
1               False  6408671
2                True   413966

Rows surviving the IS TRUE filter (GA4-eligible rows):


,n_rows_with_ga4
0,413966


- Grain violations: 0
- Total rows / distinct content / distinct clients / date span: 9841378 / 331437 / 55 /	from 2026-03-01 to	2026-03-31
- Rows with ga4_data_available IS TRUE: 413966 out of 9,841,378 total (~4.2%)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice's label (decline between the first and second half of a single month) is a
within-month proxy, not my capstone's real label (90 days of features -> 30-day future
outcome, per ML-03). A 14-day feature half and 16-day target half are both short enough
that ordinary week-to-week or mid-month noise -- not a genuine sustained decline -- could
easily produce a positive label. This slice cannot distinguish a real decline from a
short-lived dip, and it says nothing about causation. It exists only to prove the
mechanics -- grain, availability, and the leakage trap -- on real warehouse data.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.